# IMDB Movie Review — Binary Text Classification

**Goal:** classify movie reviews as positive (1) or negative (0) using a feedforward neural network.  
**Dataset:** IMDB — 50 000 reviews, top 10 000 most-frequent words, pre-encoded as integer sequences.

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print('TensorFlow version:', tf.__version__)

## Step 2 — Load the IMDB Dataset

In [ ]:
NUM_WORDS = 10_000  # keep only the 10k most frequent tokens

(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(
    num_words=NUM_WORDS
)

print('Train samples :', len(train_data))
print('Test  samples :', len(test_data))
print('Example review (integers):', train_data[0][:15], '...')
print('Label:', train_labels[0])  # 1 = positive, 0 = negative

## Step 3 — Preprocess: One-Hot Encoding (multi-hot vectorisation)

Each review is a variable-length list of integers.  
We convert it to a fixed-size binary vector of length 10 000:  
- index `i` = 1 if word `i` appears in the review, 0 otherwise.  

This lets us feed reviews into a plain Dense layer.

In [ ]:
def vectorize_sequences(sequences, dimension=NUM_WORDS):
    """Turn a list of integer sequences into a 2-D binary matrix."""
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0  # set indices that appear in the review
    return results

X_train_full = vectorize_sequences(train_data)   # (25000, 10000)
X_test       = vectorize_sequences(test_data)    # (25000, 10000)

y_train_full = train_labels.astype(np.float32)
y_test       = test_labels.astype(np.float32)

print('X_train_full shape:', X_train_full.shape)
print('X_test       shape:', X_test.shape)

## Step 4 — Train / Validation Split

We carve out the first 10 000 training samples as the validation set.

In [ ]:
VAL_SIZE = 10_000

X_val   = X_train_full[:VAL_SIZE]
X_train = X_train_full[VAL_SIZE:]

y_val   = y_train_full[:VAL_SIZE]
y_train = y_train_full[VAL_SIZE:]

print('Train :', X_train.shape, '| Val :', X_val.shape, '| Test :', X_test.shape)

## Step 5 — Build the Model

Architecture:
- **Input:** 10 000-dimensional binary vector
- **Hidden layer 1:** Dense(16, relu)
- **Hidden layer 2:** Dense(16, relu)
- **Output:** Dense(1, sigmoid) → probability of positive review

We use **binary cross-entropy** loss and **RMSprop** optimizer.

In [ ]:
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(NUM_WORDS,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid'),
])

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 6 — First Training Run (20 epochs)

We train for 20 epochs to observe overfitting, then choose the optimal epoch count.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=512,
    validation_data=(X_val, y_val),
    verbose=2
)

## Step 7 — Visualize Training vs Validation Metrics

In [ ]:
def plot_history(history):
    """Plot loss and accuracy curves for train and validation."""
    epochs = range(1, len(history.history['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # ── Loss ──────────────────────────────────────────────
    axes[0].plot(epochs, history.history['loss'],     'bo-', label='Training loss')
    axes[0].plot(epochs, history.history['val_loss'], 'rs-', label='Validation loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (binary cross-entropy)')
    axes[0].legend()

    # ── Accuracy ───────────────────────────────────────────
    axes[1].plot(epochs, history.history['accuracy'],     'bo-', label='Training accuracy')
    axes[1].plot(epochs, history.history['val_accuracy'], 'rs-', label='Validation accuracy')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history)

### Observation

The training loss decreases monotonically, but the **validation loss starts rising** after a few epochs — a clear sign of **overfitting**.  
The optimal number of epochs is roughly where the validation loss is at its minimum.  
We will use that epoch count to retrain a fresh model from scratch.

In [ ]:
# Find the epoch with minimum validation loss
val_losses = history.history['val_loss']
best_epoch = int(np.argmin(val_losses)) + 1  # +1 because epochs are 1-indexed
print(f'Best epoch (lowest val_loss): {best_epoch}')

## Step 8 — Retrain on Full Training Data for the Optimal Number of Epochs

Now that we know the right number of epochs, we retrain on **all 25 000 training samples** (train + val) for that many epochs.  
This maximises the data used for the final model before testing.

In [ ]:
tf.random.set_seed(42)

# Fresh model with the same architecture
final_model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(NUM_WORDS,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid'),
])

final_model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Retrain on full training set (train + val) for best_epoch epochs
final_history = final_model.fit(
    X_train_full, y_train_full,
    epochs=best_epoch,
    batch_size=512,
    verbose=2
)

## Step 9 — Evaluate on the Test Set

In [ ]:
test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)

print('=== Final Test Set Evaluation ===')
print(f'Loss     : {test_loss:.4f}')
print(f'Accuracy : {test_acc * 100:.2f} %')

## Step 10 — Results Analysis

| Metric | Value |
|--------|-------|
| Optimal epochs | `best_epoch` |
| Test loss (binary cross-entropy) | *see above* |
| Test accuracy | *see above* |

### Key takeaways

- **Overfitting** was visible from around epoch 4–5: training loss continued to drop while validation loss increased. Training for 20 epochs degraded generalisation.
- **Early stopping** at the optimal epoch gives a model that generalises well (~87–88 % accuracy on the test set is typical for this architecture).
- The **one-hot / multi-hot encoding** loses word-order information, yet the model still performs well, which shows that bag-of-words features carry strong sentiment signal.
- More advanced models (embeddings + LSTM, or Transformers) would push accuracy above 90 % by exploiting word order.